# ProVe-Arabic — Dataset Construction

Generation of the evaluation and annotation datasets. Referenced by Appendix A.8.

**Contents:** the Arabic gold-set candidate sheet (with identifier-property, classification-property,
reachability, and diversity filters, per Section 3.6.3), and the property-stratified
training-target review sheets (Section 3.6.2).

**Requirements:** Colab with a **T4 GPU**. The model and datasets download automatically on first run. **Run cell 1, restart the runtime when prompted, then continue.**

**This notebook is self-contained**: the setup cells below define every function it uses, so it
can be run on its own and in any order relative to the other notebooks.

## 1 — Setup  *(restart the runtime after the install cell)*

These cells reproduce the pipeline modules this notebook depends on, so it runs standalone.

In [ ]:
# Pinned environment
!pip install -q --force-reinstall --no-deps "transformers==4.46.3"
!pip install -q pysbd camel-tools beautifulsoup4 lxml requests sentencepiece protobuf sacrebleu

In [ ]:
# Fixed seeds
import os, random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
GEN = torch.Generator(); GEN.manual_seed(SEED)      # passed to DataLoader so shuffling is fixed

# For bit-identical GPU results, uncomment the two lines below BEFORE any CUDA call
# They force deterministic kernels, but slow training and raise errors for ops that have no deterministic implementation
# Seeds alone give reproducible convergence
# os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
# torch.use_deterministic_algorithms(True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| seed:', SEED)

# Datasets.
# Downloaded once from a public repository and cached locally in PD.
from huggingface_hub import hf_hub_download
import shutil, os

DATA_REPO = 'ammarikhan003/prove-arabic-data'
PD = '/content/data'; os.makedirs(PD, exist_ok=True)

FILES = ['arabic_train_pairs_fixed.jsonl',                        # verbaliser training corpus (7,607 pairs)
         'val_fixed.json',                                         # held-out set for chrF
         'arabic_gold_candidates - arabic_gold_candidates.csv',     # annotated Arabic gold set (60 items)
         'wtr_claim_features.json',                                # cached claim-level feature vectors
         'gold_features.json',
         'ar_label_cache.json']                                     # cached gold-set feature vectors

for f in FILES:
    try:
        p = hf_hub_download(repo_id=DATA_REPO, filename=f, repo_type='dataset')
        shutil.copy(p, f'{PD}/{f}')
    except Exception as e:
        print('could not fetch', f, '->', type(e).__name__)
print('data ready in', PD)
for f in sorted(os.listdir(PD)): print('  ', f)

### Stage B module

In [ ]:
!pip install -q pysbd camel-tools beautifulsoup4 lxml requests
import re, requests, pysbd
from urllib.parse import urlsplit, urlunsplit, quote
from bs4 import BeautifulSoup

# CAMeL normalisation (real role), regex fallback
try:
    from camel_tools.utils.normalize import normalize_alef_ar, normalize_alef_maksura_ar, normalize_teh_marbuta_ar
    from camel_tools.utils.dediac import dediac_ar
    def normalize_ar(t): return normalize_teh_marbuta_ar(normalize_alef_maksura_ar(normalize_alef_ar(dediac_ar(t))))
except Exception:
    def normalize_ar(t):
        t=re.sub(r'[\u064B-\u0652\u0670\u0640]','',t); t=re.sub(r'[إأآا]','ا',t)
        return t.replace('ى','ي').replace('ة','ه')

# language routing
AR=re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]')
def arabic_ratio(t):
    L=[c for c in t if c.isalpha()]
    return sum(bool(AR.match(c)) for c in L)/len(L) if L else 0.0
def detect_lang(t): return 'ar' if arabic_ratio(t)>=0.4 else 'en'

# fetch
HEADERS={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
         '(KHTML, like Gecko) Chrome/124.0 Safari/537.36 ProVe-Arabic-research/1.0',
         'Accept-Language':'ar,en;q=0.9',
         'Accept':'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'}
def _safe(u):
    p=urlsplit(u); return urlunsplit((p.scheme,p.netloc,quote(p.path),p.query,p.fragment))
def fetch_html(url):
    r=requests.get(_safe(url),headers=HEADERS,timeout=25)
    r.raise_for_status(); r.encoding=r.apparent_encoding or r.encoding
    return r.text

# clean
JUNK_TAGS=['script','style','noscript','nav','footer','header','aside','form','button','svg','iframe','figure','sup']
BLOCKS=['p','li','h1','h2','h3','h4','h5','h6','td','th','blockquote','caption','dd','dt']
LISTTOGGLE=re.compile(r'^\s*القائمة\s*\.{2,}\s*')
PUNCT_FIX=[(re.compile(r'\s+([.,؛،:!؟…\)\]])'),r'\1'),(re.compile(r'([(\[])\s+'),r'\1'),(re.compile(r'\s{2,}'),' ')]
def tidy_spacing(t):
    for p,r in PUNCT_FIX: t=p.sub(r,t)
    return t.strip()
def clean_to_text(html):
    soup=BeautifulSoup(html,'lxml')
    for t in soup(JUNK_TAGS): t.decompose()
    seen,chunks=set(),[]
    for b in soup.find_all(BLOCKS):
        txt=tidy_spacing(LISTTOGGLE.sub('',b.get_text(' ',strip=True)))
        if not txt or txt in seen: continue
        seen.add(txt)
        if txt[-1] not in '.!?؟…؛،:': txt+='.'
        chunks.append(txt)
    text='\n'.join(chunks)
    if len(text)<200: text=tidy_spacing(soup.get_text('\n',strip=True))
    return re.sub(r'\n{2,}','\n',text).strip()

# segment (pysbd + terminal merge)
TERMINALS='.؟?!…؛'
_seg={}
def _segmenter(lang):
    if lang not in _seg: _seg[lang]=pysbd.Segmenter(language=('ar' if lang=='ar' else 'en'),clean=False)
    return _seg[lang]
def segment(text,lang,min_chars=20):
    seg=_segmenter(lang); raw=[]
    for line in text.split('\n'):
        line=line.strip()
        if line: raw+=[s.strip() for s in seg.segment(line) if s.strip()]
    merged,buf=[],''
    for s in raw:
        buf=f'{buf} {s}'.strip() if buf else s
        if buf[-1] in TERMINALS: merged.append(buf); buf=''
    if buf: merged.append(buf)
    seen,out=set(),[]
    for s in merged:
        if s in seen or sum(c.isalpha() for c in s)<5 or len(s)<min_chars: continue
        seen.add(s); out.append(s)
    return out

print("Stage B module ready.")

### Stage A — verbaliser

Used to generate the draft Arabic claims the annotator then corrects.

In [ ]:
# RETRAIN = False  ->  load the trained model from the Hugging Face Hub (seconds). DEFAULT.
# RETRAIN = True   ->  rebuild it from the training corpus (~15 min on a T4).

# Training writes to a SEPARATE directory, so retraining never overwrites the
# reference weights. Both routes define the same verbalise() function.

RETRAIN = False

MODEL_REPO = 'ammarikhan003/prove-arabic-mt5'
RETRAIN_DIR = '/content/mt5_retrained'

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if not RETRAIN:
    TOK = AutoTokenizer.from_pretrained(MODEL_REPO)
    verbaliser = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO).to(device).eval()
    print('verbaliser LOADED from', MODEL_REPO)
else:
    import json
    from torch.utils.data import DataLoader
    from transformers import get_linear_schedule_with_warmup

    TOK = AutoTokenizer.from_pretrained('google/mt5-small')
    verbaliser = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small').to(device)

    pairs = [json.loads(l) for l in open(f'{PD}/arabic_train_pairs_fixed.jsonl')]
    print('training pairs:', len(pairs))

    def collate(batch):
        enc = TOK([x['input'] for x in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=64)
        lab = TOK(text_target=[x['target'] for x in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=96)['input_ids']
        lab[lab == TOK.pad_token_id] = -100      # mask padding out of the loss
        enc['labels'] = lab
        return enc

    loader = DataLoader(pairs, batch_size=8, shuffle=True,
                        collate_fn=collate, generator=GEN)   # GEN fixes the shuffle order
    opt = torch.optim.AdamW(verbaliser.parameters(), lr=3e-4)
    steps = len(loader) * 3
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)

    verbaliser.train()
    for ep in range(3):
        tot = 0.0
        for i, b in enumerate(loader):
            b = {k: v.to(device) for k, v in b.items()}
            loss = verbaliser(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(verbaliser.parameters(), 1.0)   # prevents loss spikes
            opt.step(); sched.step(); opt.zero_grad()
            tot += loss.item()
            if i % 200 == 0: print(f'  ep{ep+1} step {i}/{len(loader)} loss {loss.item():.3f}')
        print(f'epoch {ep+1}/3 avg loss {tot/len(loader):.3f}')
    verbaliser.eval()
    verbaliser.save_pretrained(RETRAIN_DIR); TOK.save_pretrained(RETRAIN_DIR)
    print('verbaliser RETRAINED and saved to', RETRAIN_DIR)

def verbalise(subj, prop, obj):
    """Triple -> Arabic sentence. Beam search for fluency; the two repetition
    constraints suppress a failure mode where a phrase repeated within one sentence."""
    enc = TOK(f'{subj} | {prop} | {obj}', return_tensors='pt',
              truncation=True, max_length=64).to(device)
    with torch.no_grad():
        o = verbaliser.generate(**enc, max_length=96, num_beams=4,
                                no_repeat_ngram_size=3, repetition_penalty=1.2)
    return TOK.decode(o[0], skip_special_tokens=True)

print('smoke test:', verbalise('دوغلاس آدمز', 'مكان الولادة', 'كامبريدج'))

## 2 — Arabic gold evaluation set: candidate sheet

SPARQL retrieval of referenced statements, then filtering. Identifier-type properties are excluded
because their objects are database identifiers pointing at registries rather than prose, which no
text-entailment pipeline can verify from document content.

References are
checked for reachability and prose content so annotators are not asked to label dead links.
Diversity caps limit pairs sharing a reference or subject.

In [ ]:
# Run AFTER Stage B module + Stage A (verbalise/mt5_fixed) are loaded.

import re, time, random, requests, pandas as pd
from collections import Counter

random.seed(42)
UA = {'User-Agent': 'ProVe-Arabic-thesis/1.0 (student project)'}

SEEDS = [
    'Q42', 'Q937', 'Q7251', 'Q935', 'Q76', 'Q1035', 'Q1339', 'Q307',  # people
    'Q84', 'Q90', 'Q60', 'Q64', 'Q1490', 'Q65', 'Q220',                # cities
    'Q95', 'Q312', 'Q2283',                                          # companies
    'Q30', 'Q145', 'Q183', 'Q142', 'Q17'                              # countries
]

values = ' '.join(f'wd:{q}' for q in SEEDS)
query = f"""SELECT ?s ?sAr ?sEn ?p ?pAr ?pEn ?o ?oAr ?oEn ?ref WHERE {{
  VALUES ?s {{ {values} }}
  ?s ?pc ?st . ?st ?ps ?o .
  ?st prov:wasDerivedFrom ?rn . ?rn pr:P854 ?ref .
  ?prop wikibase:claim ?pc ; wikibase:statementProperty ?ps ; a wikibase:Property . BIND(?prop AS ?p)
  OPTIONAL {{ ?s rdfs:label ?sAr FILTER(lang(?sAr)="ar") }}
  OPTIONAL {{ ?s rdfs:label ?sEn FILTER(lang(?sEn)="en") }}
  OPTIONAL {{ ?prop rdfs:label ?pAr FILTER(lang(?pAr)="ar") }}
  OPTIONAL {{ ?prop rdfs:label ?pEn FILTER(lang(?pEn)="en") }}
  OPTIONAL {{ ?o rdfs:label ?oAr FILTER(lang(?oAr)="ar") }}
  OPTIONAL {{ ?o rdfs:label ?oEn FILTER(lang(?oEn)="en") }}
}} LIMIT 2000"""

r = requests.get('https://query.wikidata.org/sparql', params={'query': query, 'format': 'json'}, headers=UA, timeout=120)
rows = r.json()['results']['bindings']
print('raw statements returned:', len(rows))

def val(b, k):
    return b[k]['value'] if k in b else ''

def is_uri(b, k):
    return k in b and b[k]['value'].startswith('http')

def clean_date(v):
    m = re.match(r'^([+-]?\d{3,4})-\d\d-\d\dT', v)  # 1624-01-01T00:00:00Z -> 1624
    return m.group(1).lstrip('+') if m else v

ID_HINT = re.compile(r'\b(id|identifier|code)\b', re.I)
PROP_BLOCK = {'P31', 'P279'}  # instance-of / subclass-of

cands, seen = [], set()
dropped_en = 0

for b in rows:
    p_en = val(b, 'pEn')
    p_ar = val(b, 'pAr')
    pid = val(b, 'p').rsplit('/', 1)[-1]

    if pid in PROP_BLOCK:
        continue
    if ID_HINT.search(p_en) or 'معرف' in p_ar or 'مُعرِّف' in p_ar:
        continue

    # --- object resolution ---
    o_ar_label = val(b, 'oAr')
    o_is_literal = not is_uri(b, 'o')
    o_literal = val(b, 'o') if o_is_literal else ''

    # drop entity objects with no Arabic label (English-leak rows); keep literals
    if not o_is_literal and not o_ar_label:
        dropped_en += 1
        continue

    o_ar = clean_date(o_ar_label or o_literal)
    o_en = clean_date(val(b, 'oEn') or o_literal or o_ar_label)

    # --- subject / property ---
    s_ar = val(b, 'sAr') or val(b, 'sEn')
    s_en = val(b, 'sEn') or val(b, 'sAr')
    p_ar2 = p_ar or p_en
    ref = val(b, 'ref')

    if not (s_ar and p_ar2 and o_ar and ref):
        continue

    key = (s_ar, p_ar2, o_ar, ref)
    if key in seen:
        continue
    seen.add(key)
    cands.append({'s_ar': s_ar, 'p_ar': p_ar2, 'o_ar': o_ar, 's_en': s_en, 'p_en': p_en or p_ar, 'o_en': o_en, 'ref': ref})

print(f'candidates after filters: {len(cands)}  (dropped {dropped_en} English-object rows)')

# reachability + diversity caps
random.shuffle(cands)
TARGET, MAX_PER_REF, MAX_PER_SUBJECT = 60, 2, 3
kept = []
ref_ct, subj_ct = Counter(), Counter()
fail = {'error': 0, 'too_few': 0, 'capped': 0}

for c in cands:
    if len(kept) >= TARGET:
        break
    if ref_ct[c['ref']] >= MAX_PER_REF or subj_ct[c['s_ar']] >= MAX_PER_SUBJECT:
        fail['capped'] += 1
        continue
    try:
        t = clean_to_text(fetch_html(c['ref']))
        passages = segment(t, detect_lang(t))
    except Exception:
        fail['error'] += 1
        continue
    if len(passages) >= 3:
        c['n_passages'] = len(passages)
        kept.append(c)
        ref_ct[c['ref']] += 1
        subj_ct[c['s_ar']] += 1
    else:
        fail['too_few'] += 1
    time.sleep(0.3)

print(f"kept: {len(kept)} | errors: {fail['error']} | too few: {fail['too_few']} | skipped for caps: {fail['capped']}")
print(f"diversity -> unique references: {len(ref_ct)} | unique subjects: {len(subj_ct)}")

# verbalise -> draft claim, assemble sheet
sheet = []
for c in kept:
    claim = verbalise(c['s_ar'], c['p_ar'], c['o_ar'])
    sheet.append({
        'subject': c['s_ar'],
        'property': c['p_ar'],
        'object': c['o_ar'],
        'reference_url': c['ref'],
        'english_gloss': f"{c['s_en']} | {c['p_en']} | {c['o_en']}",
        'draft_arabic_claim': claim,
        'corrected_claim': '',
        'verdict (supports/refutes/NEI)': '',
        'notes': ''
    })

df = pd.DataFrame(sheet)
out = f'{PD}/arabic_gold_candidates.csv'
df.to_csv(out, index=False, encoding='utf-8-sig')
print(f"\nwrote {len(df)} candidate rows -> {out}")

## 3 — Training-target review: batch 1

Property-stratified sample of the most frequent relational properties.

In [ ]:
import os, json, random, glob, pandas as pd
from collections import defaultdict

random.seed(42)

# File path safety check for training pairs
train_pairs_path = f'{PD}/arabic_train_pairs_fixed.jsonl' if os.path.exists(f'{PD}/arabic_train_pairs_fixed.jsonl') else f'{PD}/arabic_train_pairs.jsonl'
pairs = [json.loads(l) for l in open(train_pairs_path, encoding='utf-8')]

# properties we've ALREADY judged — skip them
EXCLUDE = {
    'النوع الفني', 'نوع الكهرباء', 'نوع النجم المتغير', 'نوع الشخصية', 'نوع إمدادات المياه',
    'المالك', 'المجموعة العرقية',              # already fixed
    'المشغل', 'له جزء أو أجزاء', 'الناشر',     # already cleared as correct
    'سبقه', 'تبعه'                             # already logged as a limitation
}

def parse(i):
    p = [x.strip() for x in i.split('|')]
    return (p + ['', '', ''])[:3]

def entity_like(o):
    o = o.strip()
    a = sum(c.isalpha() for c in o)
    d = sum(c.isdigit() for c in o)
    return a >= 2 and a >= d

# WDV english source, for disambiguating transliterated names
cands = sorted(glob.glob(f'{PD}/*wdv*') + glob.glob(f'{PD}/*WDV*'))
wdv = None
if cands:
    p = cands[0]
    wdv = [json.loads(l) for l in open(p, encoding='utf-8')] if p.endswith('.jsonl') else (
        json.load(open(p, encoding='utf-8')) if p.endswith('.json') else pd.read_csv(p).to_dict('records')
    )
    if isinstance(wdv, dict):
        wdv = list(wdv.values())

EN_FIELD = next(
    (k for k in wdv[0] if 'verbal' in k.lower() and 'unk' in k.lower()),
    next((k for k in wdv[0] if 'verbal' in k.lower()), None)
) if (wdv and len(wdv) == len(pairs)) else None

by = defaultdict(list)
for idx, ex in enumerate(pairs):
    s, p, o = parse(ex['input'])
    en = str(wdv[idx].get(EN_FIELD, '')) if (wdv and EN_FIELD) else ''
    by[p].append((s, o, ex['target'], en))

MIN_N, K, TOP_N = 5, 3, 25  # TOP_N is the size knob (top 25 properties by frequency)
props = [
    (prop, items) for prop, items in by.items()
    if len(items) >= MIN_N and prop not in EXCLUDE
    and sum(entity_like(o) for _, o, _, _ in items) / len(items) >= 0.5
]
props.sort(key=lambda kv: len(kv[1]), reverse=True)  # rank by impact (frequency)
props = props[:TOP_N]

rows = []
for prop, items in props:
    er = sum(entity_like(o) for _, o, _, _ in items) / len(items)
    for s, o, t, en in random.sample(items, min(K, len(items))):
        rows.append({
            'property': prop,
            'prop_count': len(items),
            'entity_obj_rate': round(er, 2),
            'subject': s,
            'object': o,
            'arabic_target': t,
            'english_source': en,
            'verdict (c/i/g)': '',
            'notes': ''
        })

df = pd.DataFrame(rows)
out = f'{PD}/target_review_short.csv'
df.to_csv(out, index=False, encoding='utf-8-sig')

print(f"{df['property'].nunique()} properties, {len(df)} rows -> {out}")
print("resize with TOP_N; each property adds K rows. Excluded (already reviewed):", len(EXCLUDE))

## 4 — Training-target review: batch 2

The next tranche of properties, disjoint from batch 1 by construction.

### Sampled claim–reference pairs

Defines `sample`, required by the cell below. Included here so this notebook does not
depend on the pipeline notebook having been run in the same session.

In [ ]:
import requests

# 1. pull real referenced statements from Wikidata (seeded on well-known items)
SPARQL = """
SELECT ?item ?itemLabel ?propLabel ?value ?valueLabel ?ref WHERE {
  VALUES ?item { wd:Q42 wd:Q937 wd:Q7251 wd:Q935 }
  ?item ?p ?st .
  ?st prov:wasDerivedFrom/pr:P854 ?ref .
  ?prop wikibase:claim ?p ; wikibase:statementProperty ?ps .
  ?st ?ps ?value .
  ?prop rdfs:label ?propLabel . FILTER(LANG(?propLabel) = "ar")
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ar,en". }
}
LIMIT 50
"""
r = requests.get("https://query.wikidata.org/sparql",
                 params={"query": SPARQL, "format": "json"},
                 headers={"User-Agent": "ProVe-Arabic-research/1.0 (KCL MSc dissertation)"},
                 timeout=60)
rows = r.json()["results"]["bindings"]
print(f"{len(rows)} referenced statements returned\n")

claims = []
for b in rows:
    claims.append({
        "subj": b["itemLabel"]["value"],
        "prop": b["propLabel"]["value"],
        "obj":  b.get("valueLabel", b.get("value"))["value"],
        "ref":  b["ref"]["value"],
    })

# keep a handful with distinct reference URLs
seen, sample = set(), []
for c in claims:
    if c["ref"] in seen: continue
    seen.add(c["ref"]); sample.append(c)
    if len(sample) >= 6: break

# 2. run Stage B on each real reference, pair triple <-> candidate passages
for c in sample:
    triple = f'{c["subj"]} | {c["prop"]} | {c["obj"]}'
    print("TRIPLE :", triple)
    print("REF    :", c["ref"])
    try:
        t = clean_to_text(fetch_html(c["ref"]))
        lang = detect_lang(t)
        sents = segment(t, lang)
        print(f"         [{lang}, {len(sents)} candidate passages]")
        for s in sents[:3]:
            print("   •", s[:160])
    except Exception as e:
        print("         (fetch/segment failed:", type(e).__name__, "- dead link, PDF, or JS page)")
    print()

In [ ]:
import os, json, random, glob, pandas as pd
from collections import defaultdict

random.seed(43)  # different seed from batch 1

train_pairs_path = f'{PD}/arabic_train_pairs_fixed.jsonl' if os.path.exists(f'{PD}/arabic_train_pairs_fixed.jsonl') else f'{PD}/arabic_train_pairs.jsonl'
pairs = [json.loads(l) for l in open(train_pairs_path, encoding='utf-8')]

# everything already covered: batch-1 properties + the pre-excluded set
batch1_file = f'{PD}/target_review_short.csv'
already = set(pd.read_csv(batch1_file)['property'].unique()) if os.path.exists(batch1_file) else set()

EXCLUDE = {
    'النوع الفني', 'نوع الكهرباء', 'نوع النجم المتغير', 'نوع الشخصية', 'نوع إمدادات المياه',
    'المالك', 'المجموعة العرقية', 'المشغل', 'له جزء أو أجزاء', 'الناشر', 'سبقه', 'تبعه'
}
skip = already | EXCLUDE

def parse(i):
    p = [x.strip() for x in i.split('|')]
    return (p + ['', '', ''])[:3]

def entity_like(o):
    o = o.strip()
    a = sum(c.isalpha() for c in o)
    d = sum(c.isdigit() for c in o)
    return a >= 2 and a >= d

# WDV english source, for disambiguating transliterated names (same as batch 1)
cands = sorted(glob.glob(f'{PD}/*wdv*') + glob.glob(f'{PD}/*WDV*'))
wdv = None
if cands:
    p = cands[0]
    wdv = [json.loads(l) for l in open(p, encoding='utf-8')] if p.endswith('.jsonl') else (
        json.load(open(p, encoding='utf-8')) if p.endswith('.json') else pd.read_csv(p).to_dict('records')
    )
    if isinstance(wdv, dict):
        wdv = list(wdv.values())

EN_FIELD = next(
    (k for k in wdv[0] if 'verbal' in k.lower() and 'unk' in k.lower()),
    next((k for k in wdv[0] if 'verbal' in k.lower()), None)
) if (wdv and len(wdv) == len(pairs)) else None

by = defaultdict(list)
for idx, ex in enumerate(pairs):
    s, p, o = parse(ex['input'])
    en = str(wdv[idx].get(EN_FIELD, '')) if (wdv and EN_FIELD) else ''
    by[p].append((s, o, ex['target'], en))

MIN_N, K, NEXT_N = 5, 3, 25  # next 25 unreviewed properties by frequency (~75 rows, same size as batch 1)
props = [
    (prop, items) for prop, items in by.items()
    if len(items) >= MIN_N and prop not in skip
    and sum(entity_like(o) for _, o, _, _ in items) / len(items) >= 0.5
]
props.sort(key=lambda kv: len(kv[1]), reverse=True)
props = props[:NEXT_N]

rows = []
for prop, items in props:
    er = sum(entity_like(o) for _, o, _, _ in items) / len(items)
    for s, o, t, en in random.sample(items, min(K, len(items))):
        rows.append({
            'property': prop,
            'prop_count': len(items),
            'entity_obj_rate': round(er, 2),
            'subject': s,
            'object': o,
            'arabic_target': t,
            'english_source': en,
            'verdict (c/i/g)': '',
            'notes': ''
        })

df = pd.DataFrame(rows)
out = f'{PD}/target_review_batch2.csv'
df.to_csv(out, index=False, encoding='utf-8-sig')

print(f"{df['property'].nunique()} new properties, {len(df)} rows -> {out}")
print("excluded as already-covered:", len(skip), "(batch 1 + pre-excluded)")
if not props:
    print("NOTE: no unreviewed relational properties left above the threshold — you may be done.")